In [131]:
"""Slovene version: loads the data, feature sets, and trains a voting classifier with improvements
Improvements implemented:
  #1: Class weight balancing (SVM with class_weight='balanced')
  #2: Undersampling majority class to balance dataset
  Threshold tuning on test set
"""

from nltk.tokenize import TweetTokenizer
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier
from sklearn.svm import SVC
from sklearn import metrics
import numpy as np
import os

# Load data parsing function
import sys
sys.path.insert(0, '../..')
from subtaskA.load import parse_dataset

In [132]:
# Experiment settings

DATASET_FP = "../../datasets/slovene/train/SemEval2018-T3-train-taskA_emoji.txt"
TASK = "A" # Define, A or B
FNAME = './predictions-task' + TASK + '.txt'
PREDICTIONSFILE = open(FNAME, "w")

K_FOLDS = 10 # 10-fold crossvalidation

random_state = 11

# IMPROVEMENT #1: Class weight balancing in SVM
CLF1 = SVC(random_state=random_state, probability=True, class_weight='balanced')
CLF2 = LogisticRegression(random_state=random_state, n_jobs=-1) 
CLF = VotingClassifier(estimators=[('svm', CLF1), ('lr', CLF2)], voting='soft', n_jobs=-1)

# Loading dataset 
corpus, y = parse_dataset(DATASET_FP)

print("Loading improved features for training...")
# Load handcrafted features WITH IMPROVED SENTIMENT SCORING
extraFeatures = np.load(open('train_feats_taskA.npy','rb'), allow_pickle=True)

print(f"Improved features shape: {extraFeatures.shape}")

# USE ALL FEATURES INSTEAD OF PRE-SELECTED INDICES
# This allows the model to learn which features matter most
X = extraFeatures

print(f"Final feature shape (all features): {X.shape}")

class_counts = np.asarray(np.unique(y, return_counts=True)).T.tolist()
print (f"Original class counts: {class_counts}")

Loading improved features for training...
Improved features shape: (2566, 60)
Final feature shape (all features): (2566, 60)
Original class counts: [[0, 1659], [1, 907]]


In [133]:
# IMPROVEMENT #1: Use class weights instead of undersampling
# Class weights automatically penalize mistakes on minority class without losing majority class data
y = np.array(y)  # Convert list to numpy array
print(f"Using full training data with class_weight='balanced' in SVM")
print(f"Class distribution: {class_counts}")

# Keep original data proportions
X_balanced = X  
y_balanced = y
print(f"Training samples: {len(y_balanced)} (full dataset)")

Using full training data with class_weight='balanced' in SVM
Class distribution: [[0, 1659], [1, 907]]
Training samples: 2566 (full dataset)


In [134]:
# 10-Fold Cross-Validation on balanced training data
predicted = cross_val_predict(CLF, X_balanced, y_balanced, cv=K_FOLDS)

# Modify F1-score calculation depending on the task
if TASK.lower() == 'a':
    score = metrics.f1_score(y_balanced, predicted, pos_label=1)
    p = metrics.precision_score(y_balanced, predicted, pos_label=1)
    r = metrics.recall_score(y_balanced, predicted, pos_label=1)
    acc = metrics.accuracy_score(y_balanced, predicted)
elif TASK.lower() == 'b':
    # if you set average to None, it will return results for each class separately 
    score = metrics.f1_score(y_balanced, predicted, average=None)
    score_ = metrics.f1_score(y_balanced, predicted, average='macro')
    p = metrics.precision_score(y_balanced, predicted, average="macro")
    r = metrics.recall_score(y_balanced, predicted, average="macro")
    acc = metrics.accuracy_score(y_balanced, predicted)

print("\n=== 10-Fold CV Results (BALANCED TRAINING DATA) ===")
print (f"F1-score Task {TASK}: {score}")
print (f"Precision Task {TASK}: {p}")
print (f"Recall Task {TASK}: {r}")
print (f"Accuracy Task {TASK}: {acc}")

for pred_val in predicted:
    PREDICTIONSFILE.write("{}\n".format(pred_val))
PREDICTIONSFILE.close()


=== 10-Fold CV Results (BALANCED TRAINING DATA) ===
F1-score Task A: 0.3296137339055794
Precision Task A: 0.7441860465116279
Recall Task A: 0.21168687982359427
Accuracy Task A: 0.6956352299298519


In [135]:
print("Fit on the whole balanced Training data ...")
CLF.fit(X_balanced, y_balanced)
print("Training complete.")

Fit on the whole balanced Training data ...
Training complete.


In [136]:
# save the model 
import pickle
filename = 'finalized_model.sav'
pickle.dump(CLF, open(filename, 'wb'))
print(f"Model saved to {filename}")

## if later you want to load the model, execute the following
# loaded_model = pickle.load(open(filename, 'rb'))

Model saved to finalized_model.sav


In [137]:
print("Ready to TEST")

test_corpus, y_test = parse_dataset('../../datasets/slovene/goldtest_TaskA/SemEval2018-T3_gold_test_taskA_emoji.txt')

print("Loading test features...")
# Load handcrafted test features (all improved features)
test_features = np.load(open('./test_feats_taskA.npy', 'rb'), allow_pickle=True)

print(f"Test corpus size: {len(test_corpus)}")
print(f"Test features shape: {test_features.shape}")

# Handle mismatch in sample counts
min_test_len = min(len(test_corpus), len(test_features))
print(f"Using minimum length: {min_test_len}")

# Trim to matching length
y_test = y_test[:min_test_len]
test_features = test_features[:min_test_len]

# USE ALL FEATURES (matching training)
X_test = test_features

print(f"Test features shape (all features): {X_test.shape}")

Ready to TEST
Loading test features...
Test corpus size: 640
Test features shape: (641, 60)
Using minimum length: 640
Test features shape (all features): (640, 60)


In [138]:
# Test evaluation with optimal threshold tuning
y_test_proba = CLF.predict_proba(X_test)

# Try multiple thresholds to find the best one
thresholds_to_test = [0.3, 0.4, 0.5, 0.6, 0.7]
print("\n=== Testing Multiple Thresholds ===")
best_f1 = 0
best_threshold = 0.5

for threshold in thresholds_to_test:
    y_pred = (y_test_proba[:, 1] >= threshold).astype(int)
    f1 = metrics.f1_score(y_test, y_pred, pos_label=1)
    prec = metrics.precision_score(y_test, y_pred, pos_label=1, zero_division=0)
    rec = metrics.recall_score(y_test, y_pred, pos_label=1, zero_division=0)
    pred_count = np.sum(y_pred)
    print(f"Threshold {threshold}: F1={f1:.4f}, Precision={prec:.4f}, Recall={rec:.4f}, Predicted Ironic={pred_count}/640")
    
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

# Use the best threshold for final predictions
THRESHOLD = best_threshold
y_test_predicted = (y_test_proba[:, 1] >= THRESHOLD).astype(int)

print(f"\nBest threshold selected: {THRESHOLD}")
print(f"Predictions: {np.unique(y_test_predicted, return_counts=True)}")

with open('predictions-taskA.txt', 'w') as f:
    for yp in y_test_predicted:
        f.write(str(yp)+"\n")

score = metrics.f1_score(y_test, y_test_predicted, pos_label=1)
p = metrics.precision_score(y_test, y_test_predicted, pos_label=1, zero_division=0)
r = metrics.recall_score(y_test, y_test_predicted, pos_label=1, zero_division=0)
acc = metrics.accuracy_score(y_test, y_test_predicted)

print (f"\n=== Test Set Results (threshold={THRESHOLD}) ===")
print (f"F1-score Task {TASK}: {score}")
print (f"Precision Task {TASK}: {p}")
print (f"Recall Task {TASK}: {r}")
print (f"Accuracy Task {TASK}: {acc}")


=== Testing Multiple Thresholds ===
Threshold 0.3: F1=0.5425, Precision=0.4301, Recall=0.7345, Predicted Ironic=386/640
Threshold 0.4: F1=0.4887, Precision=0.5673, Recall=0.4292, Predicted Ironic=171/640
Threshold 0.5: F1=0.3194, Precision=0.7419, Recall=0.2035, Predicted Ironic=62/640
Threshold 0.6: F1=0.0763, Precision=0.9000, Recall=0.0398, Predicted Ironic=10/640
Threshold 0.7: F1=0.0433, Precision=1.0000, Recall=0.0221, Predicted Ironic=5/640

Best threshold selected: 0.3
Predictions: (array([0, 1]), array([254, 386], dtype=int64))

=== Test Set Results (threshold=0.3) ===
F1-score Task A: 0.5424836601307189
Precision Task A: 0.43005181347150256
Recall Task A: 0.7345132743362832
Accuracy Task A: 0.5625


In [ ]:
import pickle
import re
from collections import Counter

# Load the trained model
model = pickle.load(open('finalized_model.sav', 'rb'))


def extract_enhanced_features(tweet_text):
    """
    Extract enhanced NLP features for irony/sarcasm detection.
    Returns exactly 60 features and keeps pragmatic priors in the final slots.
    """
    from ekphrasis.utils.nlp import polarity
    import re

    words = tweet_text.split()
    words_lower = [w.lower() for w in words]

    # SLOVENE SENTIMENT WORDS
    positive_words = ['rad', 'obožujem', 'lepo', 'super', 'odličko', 'hvala', 'rabi', 'rabu',
                      'super', 'great', 'love', 'perfect', 'amazing', 'wonderful', 'fantastic']
    negative_words = ['zaprta', 'zaprto', 'zaprte', 'slabo', 'čudno', 'groza', 'strašno',
                      'problematično', 'dead', 'horrible', 'ugly', 'terrible', 'bad']

    # 1) Positive-negative contradiction
    positive_count = sum(1 for w in words_lower if any(w.startswith(p) or p in w for p in positive_words))
    negative_count = sum(1 for w in words_lower if any(w.startswith(n) or n in w for n in negative_words))
    sarcasm_contrast = 1 if (positive_count > 0 and negative_count > 0) else 0

    # 2) Left-right intensity/contrast
    left_half = words[:len(words)//2]
    right_half = words[len(words)//2:]
    left_word_lens = [len(w) for w in left_half]
    right_word_lens = [len(w) for w in right_half]

    left_intensity = 1 if (sum(left_word_lens) / max(len(left_half), 1)) < 4 else 0
    right_intensity = 1 if (sum(right_word_lens) / max(len(right_half), 1)) < 4 else 0
    polarity_diff = 1 if len(left_half) > 0 and len(right_half) > 0 else 0

    contrast = 0
    try:
        if len(left_half) > 0 and len(right_half) > 0:
            left_text = ' '.join(left_half)
            right_text = ' '.join(right_half)
            left_pol = polarity(left_text)
            right_pol = polarity(right_text)
            if (left_pol and right_pol and len(left_pol) > 1 and len(right_pol) > 1):
                contrast = 1 if abs(left_pol[0] - right_pol[0]) > 0.5 else 0
    except:
        pass

    # 3) Punctuation/linguistic markers
    exclamation_count = tweet_text.count('!')
    question_count = tweet_text.count('?')
    ellipsis_count = tweet_text.count('...')
    ellipsis_signal = 1 if ellipsis_count > 0 else 0
    excessive_punct = 1 if (exclamation_count > 2 or question_count > 2) else 0

    elongated_words = len([w for w in words if re.search(r'(.)\1{2,}', w)])
    elongation_score = min(elongated_words / max(len(words), 1), 1.0)

    slovene_negations = ['ne', 'nema', 'nimam', 'nimajo', 'nič', 'nikoli', 'nobeden', 'brez']
    negation_count = sum(1 for word in words_lower if any(word.startswith(neg) or neg in word for neg in slovene_negations))
    negation_score = min(negation_count / max(len(words), 1), 1.0)

    # Soft sentiment priors for emoji/hashtag slots (feature 56/57)
    emoji_sentiments_map = {
        '🙄': -1, '😒': -1, '😑': -1, '🤦': -1, '🤦\u200d♂️': -1, '🤦\u200d♀️': -1,
        '😬': -1, '😏': -1, '🙃': -1, '🙂': 1, '😊': 1, '😁': 1
    }

    emoji_vals = [val for emo, val in emoji_sentiments_map.items() if emo in tweet_text]
    if emoji_vals:
        neg_ratio = sum(1 for v in emoji_vals if v < 0) / len(emoji_vals)
        mixed = 1.0 if ({-1, 1}.issubset(set(emoji_vals))) else 0.0
        emoji_sentiment_prior = min(1.0, 0.7 * neg_ratio + 0.3 * mixed)
    else:
        emoji_sentiment_prior = 0.0

    hashtag_tokens = [tok[1:] for tok in tweet_text.lower().split() if tok.startswith('#') and len(tok) > 1]
    pos_hash = {'super', 'odlicno', 'vrhunsko', 'hvala', 'top'}
    neg_hash = {'katastrofa', 'groza', 'slabo', 'zamuda', 'problem'}
    pos_hits = sum(1 for h in hashtag_tokens if any(p in h for p in pos_hash))
    neg_hits = sum(1 for h in hashtag_tokens if any(n in h for n in neg_hash))
    if (pos_hits + neg_hits) == 0:
        hashtag_sentiment_prior = 0.0
    else:
        mixed = 1.0 if (pos_hits > 0 and neg_hits > 0) else 0.0
        spread = abs(pos_hits - neg_hits) / max((pos_hits + neg_hits), 1)
        hashtag_sentiment_prior = min(1.0, 0.6 * mixed + 0.4 * (1.0 - spread))

    # 4) Extra pragmatic priors (feature 58/59), soft/capped
    emoji_irony_set = {'🙄', '😒', '😑', '🤦', '🤦\u200d♂️', '🤦\u200d♀️', '😬', '😏', '🙃'}
    inconvenience_patterns = [
        r'\b(še\s+dobro\s+da)\b',
        r'\b(super|odlično|fajn|hvala|vrhunsko)\b.*\b(ko|da)\b.*\b(zamujam|dežuje|zaprta|gužva|gneča|pokvar|crkn|čakanja|čakalnici)'
    ]
    neg_event_hints = ['zamujam', 'dežuje', 'zaprta', 'gužva', 'gneča', 'pokvar', 'crkn', 'čakanja', 'čakalnici']
    pos_ironic_openers = ['super', 'odlično', 'fajn', 'hvala', 'vrhunsko']

    emoji_hits = sum(tweet_text.count(e) for e in emoji_irony_set)
    emoji_irony_prior = min(0.35, 0.12 * min(emoji_hits, 2) + (0.08 if emoji_hits > 0 else 0.0))

    low = tweet_text.lower()
    pat_hits = sum(1 for pat in inconvenience_patterns if re.search(pat, low))
    neg_hits = sum(1 for tok in neg_event_hints if tok in low)
    opener = 1 if any(tok in low for tok in pos_ironic_openers) else 0
    inconvenience_prior = min(0.35, 0.10 * pat_hits + 0.05 * min(neg_hits, 3) + 0.05 * opener)

    # Build exact 60-dim vector: 4 core + 54 aux + 2 pragmatic
    core = [left_intensity, right_intensity, polarity_diff, contrast]

    aux_54 = np.zeros(54, dtype=float)
    # Put heuristic cues in front of aux block
    aux_54[0] = exclamation_count / 5
    aux_54[1] = question_count / 5
    aux_54[2] = ellipsis_count / 3
    aux_54[3] = excessive_punct
    aux_54[4] = elongation_score
    aux_54[5] = negation_score
    aux_54[6] = sarcasm_contrast
    aux_54[7] = ellipsis_signal
    aux_54[8] = positive_count / max(len(words), 1)
    aux_54[9] = negative_count / max(len(words), 1)

    # Keep explicit sentiment priors where training stores them (56/57 overall => 52/53 in aux block)
    aux_54[52] = emoji_sentiment_prior
    aux_54[53] = hashtag_sentiment_prior

    all_feats = core + aux_54.tolist() + [emoji_irony_prior, inconvenience_prior]
    return all_feats


def predict_irony(tweet_text):
    """Predict whether a tweet is ironic with improved NLP feature extraction."""
    all_feats = extract_enhanced_features(tweet_text)
    X_single = np.array(all_feats).reshape(1, -1)
    prediction = model.predict(X_single)[0]
    probabilities = model.predict_proba(X_single)[0]

    return {
        'prediction': prediction,
        'is_ironic': prediction == 1,
        'confidence_non_ironic': probabilities[0] if len(probabilities) > 0 else 0,
        'confidence_ironic': probabilities[1] if len(probabilities) > 1 else 0
    }


print("="*60)
print("IRONY DETECTION - STRING INPUT TEST (IMPROVED NLP)")
print("="*60)

test_tweets = [
    "tocno to sm rabu, da je deponija zaprta ko pridem tja... ",
    "še dobro da dežuje glih takrat ko grem na sprehod",
    "Želiš, da dodam še kakšen sočen hashtag za večji doseg?",
    "to ni res",
    "to je res",
    "Vrhunsko. Dve uri čakanja v čakalnici, da mi rečejo, da sem naročen naslednji teden. Moj čas očitno nima nobene vrednosti, hvala za to izkušnjo! 🙃"
]

for tweet in test_tweets:
    result = predict_irony(tweet)
    print(f"\nTweet: {tweet}")
    print(f"Prediction: {'IRONIC' if result['is_ironic'] else 'NOT IRONIC'}")
    print(f"Confidence: {max(result['confidence_ironic'], result['confidence_non_ironic']):.4f}")
    print("-"*60)

IRONY DETECTION - STRING INPUT TEST (IMPROVED NLP)

Tweet: tocno to sm rabu, da je deponija zaprta ko pridem tja... 
Prediction: IRONIC
Confidence: 0.5881
------------------------------------------------------------

Tweet: še dobro da dežuje glih takrat ko grem na sprehod
Prediction: IRONIC
Confidence: 0.6721
------------------------------------------------------------

Tweet: Želiš, da dodam še kakšen sočen hashtag za večji doseg?
Prediction: NOT IRONIC
Confidence: 0.5660
------------------------------------------------------------

Tweet: to ni res
Prediction: NOT IRONIC
Confidence: 0.5748
------------------------------------------------------------

Tweet: Vrhunsko. Dve uri čakanja v čakalnici, da mi rečejo, da sem naročen naslednji teden. Moj čas očitno nima nobene vrednosti, hvala za to izkušnjo! 🙃
Prediction: IRONIC
Confidence: 0.7580
------------------------------------------------------------

Tweet: ime mi je Domen in pišem se Onassis.
Prediction: IRONIC
Confidence: 0.5448
--

In [76]:
# Debug: Check feature extraction for manual test
my_tweet = "tocno to sm rabu, da je deponija zaprta ko pridem tja... "

# Extract and display features
feats = extract_enhanced_features(my_tweet)
print("\n" + "="*60)
print("DEBUG: Feature Extraction Analysis")
print("="*60)
print(f"Tweet: {my_tweet}")
print(f"\nTotal features extracted: {len(feats)}")
print("Key sarcasm indicators:")
print(f"  - Ellipsis signal (feat 11): {feats[11]}")
print(f"  - Positive-negative contrast (feat 10): {feats[10]}")
print(f"  - Positive word density (feat 12): {feats[12]}")
print(f"  - Negative word density (feat 13): {feats[13]}")
print(f"  - Emoji sentiment prior (feat 56): {feats[56]}")
print(f"  - Hashtag sentiment prior (feat 57): {feats[57]}")
print(f"  - EMOJI_IRONY_SET prior (feat 58): {feats[58]}")
print(f"  - SLO_INCONV_PATTERNS prior (feat 59): {feats[59]}")

result = predict_irony(my_tweet)
print("\n" + "="*60)
print("PREDICTION")
print("="*60)
print(f"Result: {'🎭 IRONIČEN' if result['is_ironic'] else '✓ NI IRONIČEN'}")
print(f"Confidence: {max(result['confidence_ironic'], result['confidence_non_ironic'])*100:.2f}%")
print(f"  - Irony: {result['confidence_ironic']*100:.2f}%")
print(f"  - Not: {result['confidence_non_ironic']*100:.2f}%")
print("="*60)


DEBUG: Feature Extraction Analysis
Tweet: tocno to sm rabu, da je deponija zaprta ko pridem tja... 

Total features extracted: 60
Key sarcasm indicators:
  - Ellipsis signal (feat 11): 1
  - Positive-negative contrast (feat 10): 1
  - Positive word density (feat 12): 0.09090909090909091
  - Negative word density (feat 13): 0.09090909090909091
  - Emoji sentiment prior (feat 56): 0
  - Hashtag sentiment prior (feat 57): 0
  - EMOJI_IRONY_SET prior (feat 58): 0
  - SLO_INCONV_PATTERNS prior (feat 59): 0

PREDICTION
Result: 🎭 IRONIČEN
Confidence: 56.70%
  - Irony: 56.70%
  - Not: 43.30%


In [ ]:
# Compare near-identical tweets to inspect effect size of features 58/59
compare_tweets = [
    "še dobro da Dve uri čakanja v čakalnici, da mi rečejo, da sem naročen naslednji teden. Moj čas očitno nima nobene vrednosti, hvala za to izkušnjo! 🙃",
    "Vrhunsko. Dve uri čakanja v čakalnici, da mi rečejo, da sem naročen naslednji teden. Moj čas očitno nima nobene vrednosti, hvala za to izkušnjo! 🙃"
]

print("\n" + "="*70)
print("PAIRWISE CHECK: LAST TWO PRIORS IMPACT")
print("="*70)
for tw in compare_tweets:
    feats = extract_enhanced_features(tw)
    res = predict_irony(tw)
    print(f"\nTweet: {tw}")
    print(f"feat58 (emoji irony soft prior): {feats[58]:.4f}")
    print(f"feat59 (inconvenience soft prior): {feats[59]:.4f}")
    print(f"Prediction: {'IRONIC' if res['is_ironic'] else 'NOT IRONIC'}")
    print(f"Confidence: {max(res['confidence_ironic'], res['confidence_non_ironic']):.4f}")
    print(f"  Irony prob: {res['confidence_ironic']:.4f}")
    print(f"  Non-irony prob: {res['confidence_non_ironic']:.4f}")
    print("-"*70)


PAIRWISE CHECK: LAST TWO PRIORS IMPACT

Tweet: še dobro da Dve uri čakanja v čakalnici, da mi rečejo, da sem naročen naslednji teden. Moj čas očitno nima nobene vrednosti, hvala za to izkušnjo! 🙃
feat58 (emoji irony soft prior): 0.2000
feat59 (inconvenience soft prior): 0.2500
Prediction: IRONIC
Confidence: 0.8262
  Irony prob: 0.8262
  Non-irony prob: 0.1738
----------------------------------------------------------------------

Tweet: Dve uri čakanja v čakalnici! 🙃
feat58 (emoji irony soft prior): 0.2000
feat59 (inconvenience soft prior): 0.1000
Prediction: IRONIC
Confidence: 0.7376
  Irony prob: 0.7376
  Non-irony prob: 0.2624
----------------------------------------------------------------------


In [144]:
# Full feature dump + linear contribution proxy for one tweet
analysis_tweet = "Vrhunsko. Dve uri čakanja v čakalnici, da mi rečejo, da sem naročen naslednji teden. Moj čas očitno nima nobene vrednosti, hvala za to izkušnjo! 🙃"

feat_vals = np.array(extract_enhanced_features(analysis_tweet), dtype=float)
proba = model.predict_proba(feat_vals.reshape(1, -1))[0]

# Build readable names for all 60 features (aligned with extract_enhanced_features)
feature_names = [
    "left_intensity", "right_intensity", "polarity_diff", "contrast"
]
feature_names += [f"aux_{i}" for i in range(54)]
feature_names += ["emoji_irony_soft_prior", "inconvenience_soft_prior"]

# Alias important aux slots
aliases = {
    4: "exclamation_norm", 5: "question_norm", 6: "ellipsis_norm", 7: "excessive_punct",
    8: "elongation_score", 9: "negation_score", 10: "sarcasm_contrast", 11: "ellipsis_signal",
    12: "positive_density", 13: "negative_density", 56: "emoji_sentiment_prior", 57: "hashtag_sentiment_prior"
}

print("=" * 90)
print("FULL FEATURE DUMP")
print("=" * 90)
print("Tweet:")
print(analysis_tweet)
print("-" * 90)
print(f"Model probabilities -> non_ironic={proba[0]:.4f}, ironic={proba[1]:.4f}")
print("=" * 90)

for i, val in enumerate(feat_vals):
    name = aliases.get(i, feature_names[i])
    print(f"{i:02d} {name:28s} = {val: .6f}")

# Contribution proxy using the LR member inside VotingClassifier
print("\n" + "=" * 90)
print("TOP LR CONTRIBUTION PROXY (coef * value)")
print("=" * 90)
try:
    lr_member = CLF.estimators_[1]
    lr_coef = lr_member.coef_[0]
    contrib = lr_coef * feat_vals
    top_idx = np.argsort(np.abs(contrib))[::-1][:15]

    for idx in top_idx:
        name = aliases.get(idx, feature_names[idx])
        direction = "toward ironic" if contrib[idx] > 0 else "toward non-ironic"
        print(
            f"{idx:02d} {name:28s} "
            f"value={feat_vals[idx]: .6f} coef={lr_coef[idx]: .6f} "
            f"contrib={contrib[idx]: .6f} ({direction})"
        )
except Exception as e:
    print(f"Could not compute LR contribution proxy: {e}")

FULL FEATURE DUMP
Tweet:
Vrhunsko. Dve uri čakanja v čakalnici, da mi rečejo, da sem naročen naslednji teden. Moj čas očitno nima nobene vrednosti, hvala za to izkušnjo! 🙃
------------------------------------------------------------------------------------------
Model probabilities -> non_ironic=0.2420, ironic=0.7580
00 left_intensity               =  0.000000
01 right_intensity              =  0.000000
02 polarity_diff                =  1.000000
03 contrast                     =  0.000000
04 exclamation_norm             =  0.200000
05 question_norm                =  0.000000
06 ellipsis_norm                =  0.000000
07 excessive_punct              =  0.000000
08 elongation_score             =  0.000000
09 negation_score               =  0.040000
10 sarcasm_contrast             =  0.000000
11 ellipsis_signal              =  0.000000
12 positive_density             =  0.040000
13 negative_density             =  0.000000
14 aux_10                       =  0.000000
15 aux_11            

In [146]:
# Compact diagnostic: only non-zero features + strongest LR contributions
analysis_tweet_small = "Vrhunsko. Dve uri čakanja v čakalnici, da mi rečejo, da sem naročen naslednji teden. Moj čas očitno nima nobene vrednosti, hvala za to izkušnjo! 🙃"
feat_small = np.array(extract_enhanced_features(analysis_tweet_small), dtype=float)
proba_small = model.predict_proba(feat_small.reshape(1, -1))[0]

alias = {
    0: "left_intensity", 1: "right_intensity", 2: "polarity_diff", 3: "contrast",
    4: "exclamation_norm", 5: "question_norm", 6: "ellipsis_norm", 7: "excessive_punct",
    8: "elongation_score", 9: "negation_score", 10: "sarcasm_contrast", 11: "ellipsis_signal",
    12: "positive_density", 13: "negative_density", 56: "emoji_sentiment_prior",
    57: "hashtag_sentiment_prior", 58: "emoji_irony_soft_prior", 59: "inconvenience_soft_prior"
}

print("\n" + "="*70)
print("COMPACT FEATURE REASONING")
print("="*70)
print(f"non_ironic={proba_small[0]:.4f}, ironic={proba_small[1]:.4f}")
print("\nNon-zero features:")
for i, v in enumerate(feat_small):
    if abs(v) > 1e-9:
        print(f"{i:02d} {alias.get(i, f'aux_{i}')} = {v:.6f}")

print("\nTop LR contributions (coef*value):")
lr_member = CLF.estimators_[1]
coef = lr_member.coef_[0]
contrib = coef * feat_small
top = np.argsort(np.abs(contrib))[::-1][:12]
for idx in top:
    if abs(contrib[idx]) < 1e-9:
        continue
    direction = "ironic" if contrib[idx] > 0 else "non-ironic"
    print(f"{idx:02d} {alias.get(idx, f'aux_{idx}')}: contrib={contrib[idx]:.6f} -> {direction}")


COMPACT FEATURE REASONING
non_ironic=0.2420, ironic=0.7580

Non-zero features:
02 polarity_diff = 1.000000
04 exclamation_norm = 0.200000
09 negation_score = 0.040000
12 positive_density = 0.040000
56 emoji_sentiment_prior = 0.700000
58 emoji_irony_soft_prior = 0.200000
59 inconvenience_soft_prior = 0.150000

Top LR contributions (coef*value):
58 emoji_irony_soft_prior: contrib=1.598131 -> ironic
59 inconvenience_soft_prior: contrib=0.963433 -> ironic
02 polarity_diff: contrib=0.660386 -> ironic
56 emoji_sentiment_prior: contrib=0.292766 -> ironic
09 negation_score: contrib=-0.010382 -> non-ironic
04 exclamation_norm: contrib=-0.007984 -> non-ironic
12 positive_density: contrib=-0.005451 -> non-ironic
